# CPU 網羅検証 12 (10項目完全版)


In [ ]:
import os
import sys
import subprocess
import time
import glob
import compileall
import re

print("============================================================")
print(" CPU 網羅検証 12 (10項目完全検証)")
print("============================================================")
print(f"Python: {sys.version}")

def get_dir_size(path):
    total = 0
    if not os.path.exists(path):
        return 0.0
    for dirpath, dirnames, filenames in os.walk(path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            if not os.path.islink(fp):
                total += os.path.getsize(fp)
    return total / (1024 * 1024)

site_pkgs = "/usr/local/lib/python3.13/site-packages"
dist_pkgs = "/usr/local/lib/python3.13/dist-packages"
target_dir = site_pkgs if os.path.isdir(site_pkgs) else dist_pkgs

print(f"Target site-packages: {target_dir}")
init_size = get_dir_size(target_dir)
print(f"1. インストール前 site-packages 容量: {init_size:.2f} MB")

print("\n=== データセット探索 == /kaggle/input ===")
inputs = glob.glob("/kaggle/input/*")
print("inputs:", inputs)

whl_dir = None
model_dir = None
for d in inputs:
    sub = os.listdir(d)
    if any(f.endswith(".whl") for f in sub):
        whl_dir = d
        print(f"  -> wheelhouse 発見: {d} ({len(sub)} files)")
    if any(f.endswith(".safetensors") or f.endswith(".json") for f in sub):
        model_dir = d
        print(f"  -> model/dataset 発見: {d} ({len(sub)} files)")

whls = glob.glob(os.path.join(whl_dir, "*.whl")) if whl_dir else []
print(f"wheelhouse 内 wheel 数: {len(whls)}")

print("\n=== 2. 二回連続インストール & 所要時間計測 ===")
for trial in (1, 2):
    t0 = time.time()
    if whls:
        cmd = [sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", "-U"] + whls
    else:
        cmd = [sys.executable, "-m", "pip", "install", "-v", "vllm"]
    
    print(f"--- インストール試行 {trial} ---")
    r = subprocess.run(cmd, capture_output=True, text=True)
    dt = time.time() - t0
    print(f"  所要時間: {dt:.2f} 秒 (rc={r.returncode})")
    if r.returncode != 0:
        print("  Stderr 抜粋:", "\n".join((r.stderr or "").strip().splitlines()[-10:]))
    else:
        print("  Stdout 抜粋:", "\n".join((r.stdout or "").strip().splitlines()[-5:]))

post_size = get_dir_size(target_dir)
print(f"9. インスト_後容量: {post_size:.2f} MB (差分: +{post_size - init_size:.2f} MB)")

print("\n=== 5. パッケージ出所確認 ===")
r_list = subprocess.run([sys.executable, "-m", "pip", "list", "--format=freeze"], capture_output=True, text=True)
print("\n".join([line for line in r_list.stdout.splitlines() if any(k in line.lower() for k in ("vllm", "torch", "triton", "transformers"))]))

print("\n=== 3. インストール済みパッケージ限定 compileall ===")
for pkg in ("vllm", "torch", "triton", "transformers"):
    p = os.path.join(target_dir, pkg)
    if os.path.isdir(p):
        res = compileall.compile_dir(p, quiet=1, force=True)
        print(f"  {pkg}: compileall {'OK' if res else 'NG'}")
    else:
        print(f"  {pkg}: not found in site-packages")

print("\n=== 4 & 7. インストール済み全 .so の ldd チェック ===")
all_sos = glob.glob(os.path.join(target_dir, "**/*.so"), recursive=True)
print(f"対象 .so 総数: {len(all_sos)}")
missing_libs = set()
for so in all_sos:
    try:
        res_ldd = subprocess.run(["ldd", so], capture_output=True, text=True, timeout=3)
        for line in (res_ldd.stdout or "").splitlines():
            if "not found" in line:
                lib_name = line.split("=>")[0].strip()
                missing_libs.add((os.path.basename(so), lib_name, line.strip()))
    except Exception:
        pass

print(f"未解決共有ライブラリ (not found): {len(missing_libs)} 件")
for so_name, lib_name, detail in list(missing_libs)[:20]:
    print(f"  [{so_name}] -> {detail}")

print("\n=== 6. 別 Python プロセスでの import 検証 ===")
test_code = [
    "import sys",
    "for m in ('torch', 'transformers', 'triton', 'vllm'):",
    "    try:",
    "        mod = __import__(m)",
    "        print(f'IMPORT_OK:{m} ({getattr(mod, \"__version__\", \"unknown\")})')",
    "    except Exception as e:",
    "        print(f'IMPORT_NG:{m}: {type(e).__name__}: {e}')"
]
r_imp = subprocess.run([sys.executable, "-c", "\n".join(test_code)], capture_output=True, text=True)
print(r_imp.stdout.strip())
if r_imp.stderr.strip():
    print("Stderr:", r_imp.stderr.strip()[:500])

print("\n=== 10. pip check 新規差分 ===")
r_chk = subprocess.run([sys.executable, "-m", "pip", "check"], capture_output=True, text=True)
print(r_chk.stdout.strip() or "(pip check clean)")

print("\n============================================================")
print(" 全 10 項目検証完了")
print("============================================================")
